In [ ]:
# Malariagen API arguments for each call below.
# Replace a value to override it; keep the marker to use the call-site
# value or the method's default.
API_KEEP_CALLSITE_VALUE = '__KEEP_CALLSITE_VALUE__'
API_CALL_PARAMETERS = {'cell_3_call_1_ag3_sample_metadata': {'sample_sets': '3.0',
                                       'sample_query': None,
                                       'sample_query_options': None,
                                       'sample_indices': None},
 'cell_5_call_1_ag3_add_extra_metadata': {'data': '__KEEP_CALLSITE_VALUE__', 'on': 'sample_id'},
 'cell_7_call_1_ag3_sample_metadata': {'sample_sets': '3.0',
                                       'sample_query': None,
                                       'sample_query_options': None,
                                       'sample_indices': None},
 'cell_8_call_1_ag3_snp_calls': {'region': '3R',
                                 'sample_sets': '3.0',
                                 'sample_query': None,
                                 'sample_query_options': None,
                                 'sample_indices': None,
                                 'site_mask': None,
                                 'site_class': None,
                                 'inline_array': True,
                                 'chunks': 'native',
                                 'cohort_size': None,
                                 'min_cohort_size': None,
                                 'max_cohort_size': None,
                                 'random_seed': 42},
 'cell_9_call_1_ag3_cnv_hmm': {'region': '3R',
                               'sample_sets': '3.0',
                               'sample_query': None,
                               'sample_query_options': None,
                               'max_coverage_variance': 0.2,
                               'inline_array': True,
                               'chunks': 'native'},
 'cell_10_call_1_ag3_haplotypes': {'region': '3R',
                                   'analysis': 'gamb_colu_arab',
                                   'sample_sets': '3.0',
                                   'sample_query': None,
                                   'sample_query_options': None,
                                   'inline_array': True,
                                   'chunks': 'native',
                                   'cohort_size': None,
                                   'min_cohort_size': None,
                                   'max_cohort_size': None,
                                   'random_seed': 42}}


In [ ]:
import inspect
from memory_profiler import memory_usage


def _call_api_with_parameters(method, call_key, *args, **kwargs):
    signature = inspect.signature(method)
    bound = signature.bind_partial(*args, **kwargs)
    bound.apply_defaults()
    for name, value in API_CALL_PARAMETERS[call_key].items():
        if not (isinstance(value, str) and value == API_KEEP_CALLSITE_VALUE):
            bound.arguments[name] = value

    positional = []
    keyword = {}
    for parameter in signature.parameters.values():
        if parameter.kind is inspect.Parameter.POSITIONAL_ONLY:
            if parameter.name in bound.arguments:
                positional.append(bound.arguments[parameter.name])
        elif parameter.kind is inspect.Parameter.POSITIONAL_OR_KEYWORD:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_POSITIONAL:
            positional.extend(bound.arguments.get(parameter.name, ()))
        elif parameter.kind is inspect.Parameter.KEYWORD_ONLY:
            if parameter.name in bound.arguments:
                keyword[parameter.name] = bound.arguments[parameter.name]
        elif parameter.kind is inspect.Parameter.VAR_KEYWORD:
            keyword.update(bound.arguments.get(parameter.name, {}))
    return method(*positional, **keyword)


In [ ]:
import malariagen_data
import pandas as pd
import numpy as np

In [ ]:
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../gcs_cache"),
)
ag3

In [ ]:
df_samples = _call_api_with_parameters(ag3.sample_metadata, 'cell_3_call_1_ag3_sample_metadata', sample_sets="3.0")
df_samples

In [ ]:
df_extra = pd.DataFrame(
    {
        "sample_id": df_samples["sample_id"],
        "flavour": np.random.choice(
            ["vanilla", "chocolate", "strawberry"], len(df_samples)
        ),
    }
)
df_extra

In [ ]:
_call_api_with_parameters(ag3.add_extra_metadata, 'cell_5_call_1_ag3_add_extra_metadata', df_extra)

In [ ]:
sample_query = "flavour == 'chocolate' and country == 'Burkina Faso'"

In [ ]:
_call_api_with_parameters(ag3.sample_metadata, 'cell_7_call_1_ag3_sample_metadata', sample_query=sample_query, sample_sets="3.0")

In [ ]:
_call_api_with_parameters(ag3.snp_calls, 'cell_8_call_1_ag3_snp_calls', region="3R", sample_query=sample_query, sample_sets="3.0")

In [ ]:
_call_api_with_parameters(ag3.cnv_hmm, 'cell_9_call_1_ag3_cnv_hmm', region="3R", sample_query=sample_query, sample_sets="3.0")

In [ ]:
_call_api_with_parameters(ag3.haplotypes, 'cell_10_call_1_ag3_haplotypes', region="3R", analysis="gamb_colu_arab", sample_query=sample_query, sample_sets="3.0")